# ArcGIS Online Feature Layer → GeoPandas → Power BI

**ES:** Este cuaderno ejecuta el siguiente flujo de trabajo:
1. Obtener capas de entidades desde ArcGIS Online (usando la sesión de ArcGIS Pro)
2. Convertir a DataFrame de GeoPandas y revisar los datos
3. Subir a Power BI mediante autenticación Device Code

**Requisito:** ArcGIS Pro debe estar abierto y con sesión iniciada.

---

**EN:** This notebook performs the following workflow:
1. Fetch feature layers from ArcGIS Online (using ArcGIS Pro login session)
2. Convert to GeoPandas DataFrame and review data
3. Upload to Power BI via Device Code authentication

**Requirement:** ArcGIS Pro must be open and logged in.

## 0. Instalación e importación de paquetes / Package installation and import

In [ ]:
# 필요 시 설치
# !pip install arcgis geopandas powerbiclient

In [ ]:
import pandas as pd
import geopandas as gpd
from arcgis.gis import GIS
from powerbiclient import QuickVisualize, get_dataset_config, Report
from powerbiclient.authentication import DeviceCodeLoginAuthentication

## 1. Conexión a ArcGIS Online / Connect to ArcGIS Online

**ES:** Si ArcGIS Pro está en ejecución, `GIS("pro")` se conecta sin necesidad de credenciales adicionales.

**EN:** If ArcGIS Pro is running, `GIS("pro")` connects without requiring additional credentials.

In [ ]:
gis = GIS("pro")
print(f"Connected as: {gis.properties.user.username}")

## 2. Búsqueda y carga de capas de entidades / Search and load feature layers

In [ ]:
# 검색할 피쳐 레이어 이름 설정
LAYER_NAME = "BASE_3_MICRO_WFL"  # 원하는 레이어 이름으로 변경

search_results = gis.content.search(LAYER_NAME, "Feature Layer")
print(f"검색 결과: {len(search_results)}건")
for item in search_results:
    print(f"  - {item.title} ({item.type}, owner: {item.owner})")

In [ ]:
# 첫 번째 검색 결과의 레이어 선택
feature_layer_collection = search_results[0]
feature_layer = feature_layer_collection.layers[0]
print(f"레이어: {feature_layer.properties.name}")
print(f"피쳐 수: {feature_layer.query(return_count_only=True)}")

## 3. Vista previa del mapa / Map preview

In [ ]:
map_view = gis.map("Guatemala, GTM")
map_view.basemap = "imagery"
map_view.add_layer(feature_layer)
map_view

## 4. Conversión a GeoPandas DataFrame y revisión / Convert to GeoPandas DataFrame and review

In [ ]:
# Spatially-enabled DataFrame으로 변환
sdf = pd.DataFrame.spatial.from_layer(feature_layer)
print(f"Shape: {sdf.shape}")
print(f"Columns: {list(sdf.columns)}")
sdf.head()

In [ ]:
# GeoPandas GeoDataFrame으로 변환
gdf = sdf.spatial.to_geodataframe()
print(f"CRS: {gdf.crs}")
gdf.info()

In [ ]:
# 기본 통계 리뷰
gdf.describe()

In [ ]:
# 간단한 시각화
gdf.plot(figsize=(12, 8), edgecolor="black", linewidth=0.3, alpha=0.6)

### 4-1. Carga de capas adicionales (opcional) / Load additional layers (optional)

**ES:** Utilice las celdas a continuación si necesita comparar varias capas.

**EN:** Use the cells below if you need to compare multiple layers.

In [ ]:
# 추가 레이어 예시 (필요 시 이름 변경)
EXTRA_LAYER_NAME = "C_todo_2023_Q2__APR_Erase_New"

extra_results = gis.content.search(EXTRA_LAYER_NAME, "Feature Layer")
if extra_results:
    extra_layer = extra_results[0].layers[0]
    extra_sdf = pd.DataFrame.spatial.from_layer(extra_layer)
    extra_gdf = extra_sdf.spatial.to_geodataframe()
    print(f"Extra layer shape: {extra_gdf.shape}")
    extra_gdf.head()
else:
    print(f"'{EXTRA_LAYER_NAME}' 레이어를 찾을 수 없습니다.")

## 5. Autenticación en Power BI / Power BI authentication

**ES:** Se utiliza el flujo Device Code. Al ejecutar, se mostrará un código que debe ingresar en el navegador para autenticarse.

**EN:** Uses the Device Code flow. When executed, a code will be displayed that you must enter in the browser to authenticate.

In [ ]:
pbi_auth = DeviceCodeLoginAuthentication()
pbi_auth

## 6. Carga de datos y visualización en Power BI / Upload data and visualize in Power BI

In [ ]:
# geometry 컬럼 제거 (Power BI는 geometry를 직접 지원하지 않음)
df_for_pbi = sdf.drop(columns=["SHAPE"], errors="ignore")
print(f"Power BI에 업로드할 데이터: {df_for_pbi.shape}")
df_for_pbi.head()

In [ ]:
# Power BI QuickVisualize로 빠른 리포트 생성
PBI_visualize = QuickVisualize(
    get_dataset_config(df_for_pbi, "ArcGIS_FeatureLayer_Report"),
    auth=pbi_auth
)
PBI_visualize

## 7. Exportar a Excel (opcional) / Export to Excel (optional)

In [ ]:
# 필요 시 Excel로도 저장
# output_path = "output_feature_data.xlsx"
# df_for_pbi.to_excel(output_path, index=False)
# print(f"저장 완료: {output_path}")